# Setup

# Import libraries and load the raw dataset again. Feature
# engineering should always start from raw data, NOT from the EDA
# notebook's DataFrame — keeps this notebook reproducible.

In [19]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
import os

os.makedirs("../data/processed", exist_ok=True)

df = pd.read_csv("../data/raw/creditcard.csv")
print("Loaded:", df.shape)

Loaded: (284807, 31)


# Drop Duplicates

# EDA showed duplicate rows. Duplicates leak information between
# train and test (same row appears in both) — inflating metrics.
# Droping them BEFORE the split.

In [20]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)

print(f"Dropped {before - after} duplicate rows")
print(f"New shape: {df.shape}")
print(f"Fraud count after dedup: {df['Class'].sum()}")

Dropped 1081 duplicate rows
New shape: (283726, 31)
Fraud count after dedup: 473


# Engineer Hour Feature

# EDA showed fraud spikes at 2–4 AM. Convert raw Time (seconds) to
# Hour-of-day so the model can learn this pattern. Then drop Time
# (its information is now captured by Hour).

In [21]:
df["Hour"] = (df["Time"] / 3600) % 24
df = df.drop(columns=["Time"])

print("Hour feature added. Time dropped.")
print(df[["Hour", "Amount", "Class"]].head())

Hour feature added. Time dropped.
       Hour  Amount  Class
0  0.000000  149.62      0
1  0.000000    2.69      0
2  0.000278  378.66      0
3  0.000278  123.50      0
4  0.000556   69.99      0


# Scale Amount & Hour

# V1-V28 are already PCA-scaled. Only Amount and Hour are raw.
# Use RobustScaler (not StandardScaler) because Amount has extreme
# outliers — StandardScaler would squish everything else.



# Why RobustScaler: Uses median and IQR instead of mean and std. Outliers don't distort the scale. Critical for fraud amounts.

In [22]:
scaler = RobustScaler()

df["Amount_scaled"] = scaler.fit_transform(df[["Amount"]])
df["Hour_scaled"]   = scaler.fit_transform(df[["Hour"]])

df = df.drop(columns=["Amount", "Hour"])
print("Scaled columns added, raw columns dropped.")
print(df[["Amount_scaled", "Hour_scaled"]].describe())

Scaled columns added, raw columns dropped.
       Amount_scaled    Hour_scaled
count  283726.000000  283726.000000
mean        0.924387      -0.054027
std         3.482123       0.669528
min        -0.305938      -1.718903
25%        -0.228063      -0.505217
50%         0.000000       0.000000
75%         0.771937       0.494783
max       356.962314       1.029649


# Separate Features & Target

# Split X (features) from y (target). Keep all V1-V28 + the two
# scaled features. y is the fraud label (0/1).

In [23]:
X = df.drop(columns=["Class"])
y = df["Class"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Fraud %:", round(y.mean() * 100, 4))

X shape: (283726, 30)
y shape: (283726,)
Fraud %: 0.1667


# Train/Test Split (Stratified)

# Stratified split keeps the same fraud ratio in train and test.
# Without stratification, the test set might have 0 frauds by chance.
# Use 80/20 split. random_state=42 for reproducibility.

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("\nTrain fraud count:", y_train.sum(), f"({y_train.mean()*100:.4f}%)")
print("Test  fraud count:", y_test.sum(),  f"({y_test.mean()*100:.4f}%)")

Train shape: (226980, 30)
Test shape : (56746, 30)

Train fraud count: 378 (0.1665%)
Test  fraud count: 95 (0.1674%)


# Save Processed Data

# Save train/test splits as CSV. Switched from parquet because
# pyarrow isn't installed. CSV is slower and larger but 100%
# portable — works everywhere with zero dependencies.

In [30]:

import os
os.makedirs("../data/processed", exist_ok=True)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv",   index=False)
y_train.to_frame("Class").to_csv("../data/processed/y_train.csv", index=False)
y_test.to_frame("Class").to_csv("../data/processed/y_test.csv",   index=False)

print("Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print("Location: ../data/processed/")

Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv
Location: ../data/processed/


# Verification


# Reload saved CSVs and verify shapes match. Production habit —
# never trust that saving worked without checking.

In [31]:
X_train_loaded = pd.read_csv("../data/processed/X_train.csv")
X_test_loaded  = pd.read_csv("../data/processed/X_test.csv")
y_train_loaded = pd.read_csv("../data/processed/y_train.csv")
y_test_loaded  = pd.read_csv("../data/processed/y_test.csv")

assert X_train_loaded.shape == X_train.shape
assert X_test_loaded.shape  == X_test.shape
assert len(y_train_loaded)  == len(y_train)
assert len(y_test_loaded)   == len(y_test)

print("✅ All shapes match. Data ready for modeling.")
print("Final features:", X_train_loaded.shape[1])
print("Feature names:", X_train_loaded.columns.tolist())

✅ All shapes match. Data ready for modeling.
Final features: 30
Feature names: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount_scaled', 'Hour_scaled']


# Feature Engineering Summary

**What we did:**
1. Dropped duplicate rows (prevents train/test leakage).
2. Engineered `Hour` from raw `Time` (captures 2–4 AM fraud spike).
3. Scaled `Amount` and `Hour` with **RobustScaler** (handles outliers).
4. Kept all V1–V28 (already PCA-transformed).
5. Stratified 80/20 train/test split (preserves 0.172% fraud ratio).

**Why it matters:**
- Stratified split ensures test set has fraud samples.
- RobustScaler prevents outlier distortion.
- Hour feature gives the model time-of-day signal.

**Saved artifacts:**
- `data/processed/X_train.csv`
- `data/processed/X_test.csv`
- `data/processed/y_train.csv`
- `data/processed/y_test.csv`

**Next:** `03_model_training.ipynb` — train baseline Logistic Regression,
then XGBoost with `scale_pos_weight`, evaluate with AUC-PR + Recall + F1.